# Student RAG Project - Guided Implementation

## Welcome!

In this project, you'll build a **RAG (Retrieval-Augmented Generation)** system that can answer questions about your documents.

### What You'll Learn:
- ✅ File I/O (reading documents)
- ✅ String manipulation (text chunking)
- ✅ Functions and parameters
- ✅ Lists and dictionaries
- ✅ Loops and conditionals
- ✅ Basic calculations and statistics

### What's Provided for You:
- ✅ Embedding model (converts text to numbers)
- ✅ Vector database (stores and searches embeddings)
- ✅ LLM connection (generates answers)

### Your Tasks:
You'll complete **TODO sections** marked with `# TODO:` comments.

Let's get started! 🚀

---
## Setup: Import Libraries

In [39]:
#!pip install chromadb
#!pip install sentence_transformers

In [40]:
# Import the pre-built helper module
from rag_helpers import (
    EmbeddingModel,
    VectorDatabase,
    LLM,
    Timer,
    print_separator,
    print_search_results,
    print_rag_answer,
    check_setup
)

# Import standard Python libraries you'll use
from pathlib import Path
from typing import List, Dict
import json

# Check if everything is installed correctly
check_setup()

Checking setup...
✓ chromadb is installed
✓ sentence_transformers is installed
✓ requests is installed

✓ All required packages are installed!
You're ready to start!


True

---
## Configuration

Set up the basic settings for your RAG system.

In [41]:
# TODO: Change this to point to YOUR documents folder
DOCS_FOLDER = "./mydocs"

# Chunking settings (you can experiment with these!)
CHUNK_SIZE = 500      # How many characters per chunk
OVERLAP = 50          # How many characters overlap between chunks

# How many results to retrieve for each query
TOP_K = 3

print(f"Configuration:")
print(f"  Documents folder: {DOCS_FOLDER}")
print(f"  Chunk size: {CHUNK_SIZE} characters")
print(f"  Overlap: {OVERLAP} characters")
print(f"  Top-K results: {TOP_K}")

Configuration:
  Documents folder: ./mydocs
  Chunk size: 500 characters
  Overlap: 50 characters
  Top-K results: 3


---
## TODO #1: Document Loading

**Your Task:** Write a function to load all text files from a folder.

**What to do:**
1. Loop through all `.txt` files in the folder
2. Read each file's content
3. Store the content and filename in a dictionary
4. Return a list of these dictionaries

**Python concepts:** File I/O, loops, dictionaries, lists

In [42]:
def load_documents(folder_path: str) -> List[Dict[str, str]]:
    """
    Load all text documents from a folder.

    Args:
        folder_path: Path to folder containing .txt files

    Returns:
        List of dictionaries, each containing:
        - 'content': the text content of the file
        - 'filename': the name of the file

    Example:
        [
            {'content': 'This is doc 1...', 'filename': 'doc1.txt'},
            {'content': 'This is doc 2...', 'filename': 'doc2.txt'}
        ]
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty list to store documents
    # 2. Use Path(folder_path).glob("*.txt") to find all .txt files
    # 3. For each file:
    #    - Open it with open(file_path, 'r', encoding='utf-8')
    #    - Read the content with .read()
    #    - Create a dictionary with 'content' and 'filename'
    #    - Append to your list
    # 4. Return the list

    documents = []  # Start with empty list

    # Your code here:
    folder = Path(folder_path)

    for file_path in folder.glob("*.txt"):
        # Open and read the file
        # Create a dictionary
        # Add to documents list
        # Remove this 'pass' and write your code
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        documents.append({
            'content': content,
            'filename': file_path.name
            })

    print(f"✓ Loaded {len(documents)} documents")
    return documents


# Test your function
documents = load_documents(DOCS_FOLDER)

# Display first document (if any were loaded)
if documents:
    print(f"\nFirst document: {documents[0]['filename']}")
    print(f"Content preview: {documents[0]['content'][:200]}...")
else:
    print("⚠️  No documents loaded! Check your folder path.")

✓ Loaded 10 documents

First document: Adventure_Photography.txt
Content preview: Adventure Photography:
Adventure photography captures experiences in hiking, climbing, and travel, highlighting both scenery and human spirit. 
Photographers balance documenting the moment with stayin...


---
## TODO #2: Text Chunking Function

**Your Task:** Write a function to split long text into smaller chunks with overlap.

**Why?** Long documents are too big for embeddings. We need to split them into smaller pieces.

**What to do:**
1. Start at the beginning of the text
2. Take a chunk of `chunk_size` characters
3. Move forward by `chunk_size - overlap` characters
4. Repeat until you reach the end

**Python concepts:** String slicing, loops, lists

In [43]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.

    Args:
        text: The text to split
        chunk_size: Maximum characters per chunk
        overlap: How many characters to overlap between chunks

    Returns:
        List of text chunks

    Example:
        text = "This is a long document..."
        chunks = chunk_text(text, chunk_size=100, overlap=20)
        # Returns: ['This is a long...', 'long document...']
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty list to store chunks
    # 2. Start with position = 0
    # 3. While position < len(text):
    #    - Extract chunk from position to position+chunk_size
    #    - Add chunk to list (if not empty)
    #    - Move position forward by (chunk_size - overlap)
    # 4. Return the list of chunks

    chunks = []  # Start with empty list
    position = 0  # Start at beginning

    # Your code here:
    while position < len(text):
        # Extract a chunk
        # Add to chunks list
        # Move position forward
        # Remove this 'pass' and write your code
        chunk = text[position:position + chunk_size]
        if chunk:
            chunks.append(chunk)
        position += chunk_size - overlap

    return chunks


# Test your chunking function
test_text = "Hey how you doing?" * 50  # Create a long test string
test_chunks = chunk_text(test_text, chunk_size=100, overlap=20)

print(f"Test text length: {len(test_text)} characters")
print(f"Number of chunks: {len(test_chunks)}")
print(f"\nFirst chunk: {test_chunks[0]}")
if len(test_chunks) > 1:
    print(f"Second chunk: {test_chunks[1]}")

Test text length: 900 characters
Number of chunks: 12

First chunk: Hey how you doing?Hey how you doing?Hey how you doing?Hey how you doing?Hey how you doing?Hey how yo
Second chunk: you doing?Hey how you doing?Hey how you doing?Hey how you doing?Hey how you doing?Hey how you doing?


---
## TODO #3: Process All Documents into Chunks

**Your Task:** Use your chunking function to split ALL documents into chunks and create metadata.

**What to do:**
1. Loop through each document
2. Chunk the document's content
3. For each chunk, create metadata (which file it came from, which chunk number)
4. Store everything in a list

**Python concepts:** Nested loops, dictionaries, enumerate

In [44]:
def process_documents(documents: List[Dict[str, str]],
                     chunk_size: int = 500,
                     overlap: int = 50) -> tuple:
    """
    Process all documents into chunks with metadata.

    Args:
        documents: List of document dictionaries
        chunk_size: Size of each chunk
        overlap: Overlap between chunks

    Returns:
        Tuple of (chunk_texts, chunk_metadatas)
        - chunk_texts: List of chunk strings
        - chunk_metadatas: List of metadata dictionaries
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create two empty lists: chunk_texts and chunk_metadatas
    # 2. For each document:
    #    - Get the document's content and filename
    #    - Use your chunk_text() function to split it
    #    - For each chunk (use enumerate to get index):
    #      - Add chunk text to chunk_texts
    #      - Create metadata dict with 'source' and 'chunk_id'
    #      - Add metadata to chunk_metadatas
    # 3. Return both lists as a tuple

    chunk_texts = []
    chunk_metadatas = []

    # Your code here:
    for doc in documents:
        # Get document content and filename
        # Chunk the document
        # For each chunk:
        #   - Add chunk text to chunk_texts
        #   - Create metadata dictionary
        #   - Add metadata to chunk_metadatas
        # Remove this 'pass' and write your code
        content = doc['content']
        filename = doc['filename']
        chunks = chunk_text(content, chunk_size, overlap)
        for idx, chunk in enumerate(chunks):
            chunk_texts.append(chunk)
            metadata = {
                'source': filename,
                'chunk_id': idx
            }
            chunk_metadatas.append(metadata)

    print(f"✓ Created {len(chunk_texts)} chunks from {len(documents)} documents")
    return chunk_texts, chunk_metadatas


# Process all documents
chunk_texts, chunk_metadatas = process_documents(documents, CHUNK_SIZE, OVERLAP)

# Display example
if chunk_texts:
    print(f"\nExample chunk:")
    print(f"  Source: {chunk_metadatas[0]['source']}")
    print(f"  Chunk ID: {chunk_metadatas[0]['chunk_id']}")
    print(f"  Text: {chunk_texts[0][:200]}...")

✓ Created 22 chunks from 10 documents

Example chunk:
  Source: Adventure_Photography.txt
  Chunk ID: 0
  Text: Adventure Photography:
Adventure photography captures experiences in hiking, climbing, and travel, highlighting both scenery and human spirit. 
Photographers balance documenting the moment with stayin...


---
## Pre-Built: Create Embeddings and Store in Database

This part uses the pre-built helpers. Just run these cells - no coding needed! ✨

In [45]:
# Initialize the embedding model (pre-built)
print("Initializing embedding model...")
embedder = EmbeddingModel()

# Create embeddings for all chunks (pre-built)
print("\nCreating embeddings...")
embeddings = embedder.embed_multiple(chunk_texts)
print(f"✓ Created {len(embeddings)} embeddings")

Initializing embedding model...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
✓ Model loaded!

Creating embeddings...
Embedding 22 texts...
✓ Complete!
✓ Created 22 embeddings


In [46]:
# Initialize vector database (pre-built)
print("Initializing vector database...")
vector_db = VectorDatabase()

# Add chunks to database (pre-built)
print("\nAdding chunks to database...")
vector_db.add_chunks(chunk_texts, embeddings, chunk_metadatas)

Initializing vector database...
✓ Vector database initialized
  Collection: student_rag
  Current documents: 29

Adding chunks to database...
✓ Added 22 chunks to database


In [47]:
# Initialize LLM connection (pre-built)
print("Connecting to Ollama LLM...")
llm = LLM(model="gemma3:1b-it-qat")

# Test the connection
print("\nTesting LLM connection...")
if llm.test_connection():
    print("✓ LLM is working!")
else:
    print("⚠️  LLM connection failed! Make sure Docker container is running.")

Connecting to Ollama LLM...
✓ LLM initialized: gemma3:1b-it-qat at http://127.0.0.1:11434

Testing LLM connection...
✓ LLM is working!


---
## TODO #4: RAG Query Function

**Your Task:** Write the main RAG function that ties everything together!

**What to do:**
1. Embed the user's question
2. Search the database for similar chunks
3. Build a prompt with the retrieved context
4. Ask the LLM to answer based on the context
5. Return the answer and metadata

**Python concepts:** Functions, string formatting, dictionaries

In [48]:
def rag_query(question: str, top_k: int = 3) -> Dict:
    """
    Answer a question using RAG (Retrieval-Augmented Generation).

    Args:
        question: The user's question
        top_k: How many chunks to retrieve

    Returns:
        Dictionary with:
        - 'question': the original question
        - 'answer': the LLM's answer
        - 'sources': list of source filenames
        - 'contexts': list of retrieved chunks
        - 'time': how long it took
    """

    # Start timer
    timer = Timer()
    timer.start()

    # TODO: Implement the RAG pipeline!
    # HINTS:
    # 1. Embed the question using: embedder.embed_text(question)
    # 2. Search database using: vector_db.search(query_embedding, top_k)
    # 3. Extract retrieved chunks and metadata from search results:
    #    - retrieved_chunks = results['documents'][0]
    #    - retrieved_metadata = results['metadatas'][0]
    # 4. Build context by joining chunks with newlines
    # 5. Create prompt (template below)
    # 6. Generate answer using: llm.generate_answer(prompt)
    # 7. Extract source filenames from metadata
    # 8. Return everything in a dictionary

    # Step 1: Embed question
    query_embedding = None  # Your code here
    query_embedding = embedder.embed_text(question)

    # Step 2: Search database
    results = None  # Your code here
    results = vector_db.search(query_embedding, top_k)

    # Step 3: Extract results
    retrieved_chunks = None  # Your code here
    retrieved_chunks = results['documents'][0]
    retrieved_metadata = None  # Your code here
    retrieved_metadata = results['metadatas'][0]

    # Step 4: Build context
    context = None  # Your code here (join chunks with '\n\n')
    context = '\n\n'.join(retrieved_chunks)

    # Step 5: Create prompt (use this template)
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {question}

Answer:"""

    # Step 6: Generate answer
    answer = None  # Your code here
    answer = llm.generate_answer(prompt)

    # Step 7: Extract sources
    sources = []  # Your code here (get 'source' from each metadata dict)
    for metadata in retrieved_metadata:
        sources.append(metadata['source'])

    # Stop timer
    elapsed_time = timer.stop()

    # Step 8: Return results
    return {
        'question': question,
        'answer': answer,
        'sources': sources,
        'contexts': retrieved_chunks,
        'time': elapsed_time
    }


print("✓ RAG query function defined!")

✓ RAG query function defined!


---
## Test Your RAG System!

Let's try asking some questions!

In [49]:
# Test question 1
result = rag_query("What is adventure photography?")

# Pretty print the result
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: What is adventure photography?

ANSWER:
Adventure photography captures experiences in hiking, climbing, and travel, highlighting both scenery and human spirit.

SOURCES: Adventure_Photography.txt, Nature_Wildlife.txt
TIME: 3.20 seconds


In [62]:
# Try your own question!
my_question = "How can photographers click photos without disturbing wildlife?"  # Change this!

result = rag_query(my_question)
print_rag_answer(
    result['question'],
    result['answer'],
    result['sources'],
    result['time']
)

======================== RAG ANSWER ========================

QUESTION: How can photographers click photos without disturbing wildlife?

ANSWER:
Photographers should wear neutral clothing and minimize noise to prevent startling wildlife.  Fast shutter speeds freeze motion, while continuous autofocus tracks unpredictable movement.

SOURCES: Nature_Wildlife.txt
TIME: 2.17 seconds


---
## TODO #5: Create Test Dataset

**Your Task:** Create a list of test questions to evaluate your RAG system.

**What to do:**
1. Think of 10 questions your documents can answer
2. For each question, write the expected answer
3. Store them in a structured format

**Python concepts:** Lists, dictionaries, data structures

In [51]:
# TODO: Create your test dataset!
# HINTS:
# Create a list of dictionaries
# Each dictionary should have:
#   - 'question': the test question
#   - 'expected_answer': what you expect the answer to include
#   - 'category': type of question (factual, inferential, etc.)

test_questions = [
    # Example (replace with your own!):
    {
        'question': 'What is exposure in photography?',
        'expected_answer': 'Exposure refers to the amount of light captured by the camera and is controlled by aperture, shutter speed, and ISO.',
        'category': 'definition'
    },
    # Add 9 more questions here!
    # Your code here:
    {
        'question': 'How does aperture affect a photo?',
        'expected_answer': 'Aperture affects depth of field, where a wider aperture creates a blurry background and a narrower one keeps more of the scene in focus.',
        'category': 'factual'
    },
    {
        'question': 'What is the best time of day for landscape photography?',
        'expected_answer': 'Golden hour—shortly after sunrise and before sunset—provides warm, soft lighting ideal for landscapes.',
        'category': 'factual'
    },
    {
        'question': 'How can photographers avoid disturbing wildlife?',
        'expected_answer': 'By keeping a safe distance, using telephoto lenses, and not interfering with natural animal behavior.',
        'category': 'procedural'
    },
    {
        'question': 'Why is a tripod important in long-exposure photography?',
        'expected_answer': 'A tripod stabilizes the camera to avoid motion blur during long shutter speeds.',
        'category': 'factual'
    },
    {
        'question': 'What is macro photography used for?',
        'expected_answer': 'Macro photography captures close-up details of small subjects like insects and plants.',
        'category': 'definition'
    },
    {
        'question': 'How does ISO influence image quality?',
        'expected_answer': 'Higher ISO increases brightness but also adds noise, while lower ISO provides cleaner images.',
        'category': 'factual'
    },
    {
        'question': 'What technique helps create strong visual composition?',
        'expected_answer': 'Using the rule of thirds, leading lines, and framing can improve composition.',
        'category': 'inferential'
    },
    {
        'question': 'How can natural light be modified for photography?',
        'expected_answer': 'Natural light can be shaped using reflectors, diffusers, or by changing the angle of the subject relative to the light.',
        'category': 'procedural'
    },
    {
        'question': 'What should photographers do to protect their camera equipment in harsh weather?',
        'expected_answer': 'Use weather-sealed bags, lens hoods, and moisture-absorbing materials to protect gear.',
        'category': 'procedural'
    }
]

print(f"✓ Created {len(test_questions)} test questions")
print(f"\nExample question:")
print(f"  Q: {test_questions[0]['question']}")
print(f"  Expected: {test_questions[0]['expected_answer']}")

✓ Created 10 test questions

Example question:
  Q: What is exposure in photography?
  Expected: Exposure refers to the amount of light captured by the camera and is controlled by aperture, shutter speed, and ISO.


---
## TODO #6: Calculate Evaluation Metrics

**Your Task:** Write functions to measure how well your RAG system performs.

**Python concepts:** Functions, calculations, statistics

In [52]:
def calculate_average_latency(results: List[Dict]) -> float:
    """
    Calculate average response time.

    Args:
        results: List of result dictionaries (each has 'time' field)

    Returns:
        Average time in seconds
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Extract all 'time' values from results
    # 2. Sum them up
    # 3. Divide by the number of results
    # 4. Return the average

    # Your code here:
    total_time = sum(r['time'] for r in results if 'time' in r)
    # Calculate sum and average
    return total_time / len(results) if results else 0.0

     # Replace with your calculation


def count_successful_retrievals(results: List[Dict]) -> int:
    """
    Count how many queries successfully retrieved context.

    Args:
        results: List of result dictionaries

    Returns:
        Number of successful retrievals
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Start with count = 0
    # 2. For each result:
    #    - Check if 'contexts' is not empty
    #    - If yes, increment count
    # 3. Return count

    # Your code here:
    count = 0  # Start count at 0
    # Check each result
    for result in results:
        if result.get('contexts'):
            count += 1
    # Count successful retrievals
    return count


def get_all_sources(results: List[Dict]) -> List[str]:
    """
    Get unique list of all sources used.

    Args:
        results: List of result dictionaries

    Returns:
        List of unique source filenames
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create an empty set (sets automatically keep unique values)
    # 2. For each result:
    #    - Get the 'sources' list
    #    - Add each source to the set
    # 3. Convert set to list and return

    # Your code here:
    all_sources = set()
    # Collect all sources
    for result in results:
        for source in result['sources']:
            all_sources.add(source)

    return list(all_sources)


print("✓ Evaluation functions defined!")

✓ Evaluation functions defined!


---
## TODO #7: Run Complete Evaluation

**Your Task:** Test your RAG system with all test questions and calculate metrics.

**Python concepts:** Loops, function calls, data aggregation

In [53]:
def run_evaluation(test_questions: List[Dict]) -> List[Dict]:
    """
    Run RAG system on all test questions.

    Args:
        test_questions: List of test question dictionaries

    Returns:
        List of result dictionaries
    """

    # TODO: Implement this function!
    # HINTS:
    # 1. Create empty list for results
    # 2. For each test question:
    #    - Get the question text
    #    - Call rag_query() with the question
    #    - Add result to results list
    # 3. Return results

    results = []

    # Your code here:
    for test in test_questions:
        # Get question
        # Run RAG query
        # Store result
         # Remove this 'pass' and write your code
        question = test['question']
        result = rag_query(question)
        results.append(result)

    return results


# Run evaluation on all test questions
print("Running evaluation on all test questions...\n")
all_results = run_evaluation(test_questions)

print(f"\n✓ Completed {len(all_results)} tests")

Running evaluation on all test questions...


✓ Completed 10 tests


---
## Display Results

Show the evaluation metrics and results.

In [54]:
# Calculate metrics using your functions
avg_latency = calculate_average_latency(all_results)
successful_retrievals = count_successful_retrievals(all_results)
all_sources_used = get_all_sources(all_results)
hit_rate = successful_retrievals / len(all_results) if all_results else 0

# Display metrics
print_separator("EVALUATION RESULTS")
print(f"\nTotal Questions Tested: {len(all_results)}")
print(f"Successful Retrievals: {successful_retrievals}")
print(f"Hit Rate: {hit_rate:.2%}")
print(f"Average Latency: {avg_latency:.2f} seconds")
print(f"\nSources Used: {', '.join(all_sources_used)}")
print_separator()

==================== EVALUATION RESULTS ====================

Total Questions Tested: 10
Successful Retrievals: 10
Hit Rate: 100.00%
Average Latency: 4.48 seconds

Sources Used: Camera_Basics.txt, Adventure_Photography.txt, Lighting_in_Photography.txt, Nature_Wildlife.txt, Macro_Photography.txt, Camera_Maintenance.txt, Photography_Techniques.txt, Landscape_Photography.txt


In [55]:
# Display individual results
print("\nIndividual Test Results:\n")

for i, result in enumerate(all_results, 1):
    print(f"[Test {i}]")
    print(f"Question: {result['question']}")
    print(f"Answer: {result['answer'][:200]}...")
    print(f"Sources: {', '.join(set(result['sources']))}")
    print(f"Time: {result['time']:.2f}s")
    print("-" * 60)
    print()


Individual Test Results:

[Test 1]
Question: What is exposure in photography?
Answer: Understanding the exposure triangle—aperture, shutter speed, and ISO—helps photographers control brightness, depth of field, and motion....
Sources: Camera_Basics.txt, Photography_Techniques.txt
Time: 4.26s
------------------------------------------------------------

[Test 2]
Question: How does aperture affect a photo?
Answer: Aperture determines how much light enters and affects background blur....
Sources: Camera_Basics.txt, Photography_Techniques.txt, Lighting_in_Photography.txt
Time: 2.96s
------------------------------------------------------------

[Test 3]
Question: What is the best time of day for landscape photography?
Answer: Morning and evening offer soft, warm tones....
Sources: Lighting_in_Photography.txt, Nature_Wildlife.txt, Landscape_Photography.txt
Time: 3.06s
------------------------------------------------------------

[Test 4]
Question: How can photographers avoid disturbing wild

---
## Save Your Results

Save your test results to a JSON file for your report.

In [56]:
# Save results to JSON file
results_summary = {
    'metrics': {
        'total_questions': len(all_results),
        'successful_retrievals': successful_retrievals,
        'hit_rate': hit_rate,
        'average_latency': avg_latency
    },
    'results': all_results
}

with open('evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("✓ Results saved to 'evaluation_results.json'")

✓ Results saved to 'evaluation_results.json'


---
## Congratulations! 🎉

You've successfully built a RAG system!

### What You Accomplished:
✅ Loaded documents from files  
✅ Chunked text with overlap  
✅ Created a RAG query pipeline  
✅ Built a test dataset  
✅ Calculated evaluation metrics  
✅ Generated a results report  

### Next Steps:
- Try different chunk sizes and overlaps
- Add more test questions
- Experiment with different values for `top_k`
- Analyze which questions work best
- Write up your findings in a report

### For Your Report:
1. Describe your document collection
2. Explain your chunking strategy
3. Present your evaluation metrics
4. Show examples of good and bad answers
5. Discuss what you learned

Great job! 🚀